# Week 2, Day 1 — Agent Foundations
### Reasoning Loops, Tool Calling & Raw Python Agents

Building a minimal agent from scratch — no LangChain, no LangGraph. Uses the
**Gemini API** directly (free tier — no billing needed). The goal is to see
the mechanism behind every agent framework: **an LLM in a loop that
reasons, calls tools, observes results, and decides what to do next.**



In [1]:
pip install google-genai

   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.1 MB ? eta -:--:--
   --------- ------------------------------ 0.3/1.1 MB ? eta -:--:--
   ------------------- -------------------- 0.5/1.1 MB 904.4 kB/s eta 0:00:01
   ------------------- -------------------- 0.5/1.1 MB 904.4 kB/s eta 0:00:01
   ---------------------------- ----------- 0.8/1.1 MB 901.6 kB/s eta 0:00:01
   ---------------------------------------- 1.1/1.1 MB 890.4 kB/s  0:00:01
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ----- ---------------------------------- 0.3/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 523.9 kB/s eta 0:00:03
   ---------- ----------------------------- 0.5/2.0 MB 523.9 

In [2]:
import os
os.environ["GEMINI_API_KEY"] = " "

In [11]:
import os
if not os.environ.get("GEMINI_API_KEY"):
    print("⚠️  GEMINI_API_KEY not set yet. Set it above before running the later cells.")
else:
    print("API key found — ready to go.")

API key found — ready to go.


## Task 1 — Agent Concepts & Mental Model

**Chatbot** — a single request/response loop. It reasons over conversation
text and replies. It cannot act on the world; it has no tools and no
persistent goal beyond answering the current message.

**Workflow** — a fixed sequence of steps (possibly including LLM calls)
wired together in advance by a human. The *path* is predetermined; the LLM
fills in content within steps but doesn't choose the steps themselves or
their order.

**Agent** — an LLM that is put in a loop with tools and decides, turn by
turn, *what to do next* based on what it observes, until it decides the
task is done. The control flow is not fixed in advance — it emerges from
the model's own decisions.

**What makes something "agentic"** is a combination of:
- **Autonomy** — the system chooses its own next action rather than
  following a script.
- **Tool use** — it can affect or query the outside world (APIs, files,
  calculators, search), not just generate text.
- **Multi-step planning** — it can break a goal into an unknown-in-advance
  number of sub-actions.
- **Self-correction** — it can notice a tool failed or a result was
  unexpected, and adapt (retry, pick a different tool, ask for
  clarification) instead of blindly continuing.

**The ReAct pattern** (Yao et al., 2022) interleaves *Reasoning* and
*Acting*:

```
Reason: "The user wants X. To get it I need Y."
Act:    call tool(Y)
Observe: tool returns result
Reason: "Given that result, do I have enough to answer, or do I need Z?"
Act:    call tool(Z)  [or stop and answer]
Observe: ...
...repeat until Reason concludes "I can answer now."
```

Pseudocode:
```python
while not done:
    thought, action = llm.reason(history)
    if action is None:
        done = True
        answer = thought
    else:
        observation = execute(action)
        history.append(thought, action, observation)
```

**When an agent is overkill.** If the task has a fixed, known sequence of
steps (e.g. "resize this image, then upload it"), a plain script is faster,
cheaper, and more predictable than an agent — you already know the plan, so
paying an LLM to "discover" it each time adds latency, cost, and a new
failure surface for no benefit. Likewise, if a single well-crafted prompt
already gets a reliable answer (no tool calls or multi-step lookups
needed), reaching for a loop-with-tools architecture is unnecessary
complexity. Agents earn their cost when the number and order of steps
genuinely can't be known ahead of time.

## Task 2 — Tool Calling Fundamentals

Three tools defined below with full JSON schemas: `calculator`,
`get_weather` (stub), and `read_text_file`. Each has `name`, `description`,
and `parameters` (Gemini's name for the input schema).

**Why tool descriptions matter:** the model never sees the Python
implementation — the `description` field (both tool-level and per-argument)
*is* its entire understanding of what the tool does and when to use it. A
vague description ("does math") causes two failure modes: the model may
skip the tool and hallucinate a numeric answer itself, or it may call the
*wrong* tool for the job. A specific description with an example input and
an explicit boundary ("evaluates a single arithmetic expression like
`'12 * (4 + 1)'`; does not understand word problems") sharply reduces both
wrong-tool selection and malformed-argument errors, because it gives the
model a concrete template to match the user's request against.

In [4]:
import json
import ast
import operator as op

# --- Tool schemas (what the model sees) ---
TOOLS = [
    {
        "type": "function",
        "name": "calculator",
        "description": (
            "Evaluates a single arithmetic expression using +, -, *, /, "
            "**, and parentheses, e.g. '12 * (4 + 1)'. Use this for any "
            "numeric computation. Does not understand word problems or "
            "units — convert to a plain numeric expression first."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {"type": "string", "description": "A valid arithmetic expression, e.g. '(3 + 4) * 2'"}
            },
            "required": ["expression"],
        },
    },
    {
        "type": "function",
        "name": "get_weather",
        "description": (
            "Looks up the current weather for a named city and returns "
            "temperature in Celsius and a short condition string. Stub/demo "
            "data only — not a live weather feed. City must be a real city name."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. 'Lahore' or 'Karachi'"}
            },
            "required": ["city"],
        },
    },
    {
        "type": "function",
        "name": "read_text_file",
        "description": (
            "Reads and returns the full text contents of a local .txt file "
            "given its path. Use this when the user refers to a file on disk "
            "they want summarized, searched, or quoted."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Filesystem path to a .txt file, e.g. 'notes.txt'"}
            },
            "required": ["path"],
        },
    },
]

# --- Tool implementations (the actual Python behind each schema) ---

_ALLOWED_OPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul, ast.Div: op.truediv,
    ast.Pow: op.pow, ast.USub: op.neg,
}

def _safe_eval(node):
    if isinstance(node, ast.Constant):
        if isinstance(node.value, (int, float)):
            return node.value
        raise ValueError("Only numeric constants are allowed")
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_safe_eval(node.operand))
    raise ValueError(f"Unsupported expression element: {node!r}")

def calculator(expression: str):
    tree = ast.parse(expression, mode="eval")
    return _safe_eval(tree.body)

# Stub weather data — deterministic so results are reproducible.
_FAKE_WEATHER = {
    "lahore": {"temp_c": 34, "condition": "Sunny"},
    "karachi": {"temp_c": 29, "condition": "Humid"},
    "islamabad": {"temp_c": 27, "condition": "Partly cloudy"},
    "bahawalpur": {"temp_c": 38, "condition": "Hot and dry"},
    "murree": {"temp_c": 18, "condition": "Cool, light rain"},
}

def get_weather(city: str):
    key = city.strip().lower()
    if key not in _FAKE_WEATHER:
        raise KeyError(f"No weather data available for '{city}' (stub only covers a fixed city list)")
    return {"city": city, **_FAKE_WEATHER[key]}

def read_text_file(path: str):
    with open(path, "r") as f:
        return f.read()

TOOL_FUNCTIONS = {
    "calculator": calculator,
    "get_weather": get_weather,
    "read_text_file": read_text_file,
}

print(f"{len(TOOLS)} tools defined:", [t["name"] for t in TOOLS])

3 tools defined: ['calculator', 'get_weather', 'read_text_file']


### Single tool-call round trip (send request → model picks a tool → we execute it → return the result)

This is the one-shot version of the exchange, run for real against Gemini
— the same mechanism the full loop in Task 3 repeats, shown here as a
single explicit round trip.

In [5]:
from google import genai

client = genai.Client()

interaction = client.interactions.create(
    model="gemini-3.8-flash",
    input="What is (15 * 4) + 7?",
    tools=TOOLS,
)

fc_step = next((s for s in interaction.steps if s.type == "function_call"), None)
print("Model chose tool:", fc_step.name, fc_step.arguments)

# Execute it manually
result = TOOL_FUNCTIONS[fc_step.name](**fc_step.arguments)
print("Tool result:", result)

# Send the result back and get the final answer
final_interaction = client.interactions.create(
    model="gemini-3.8-flash",
    previous_interaction_id=interaction.id,
    input=[{
        "type": "function_result",
        "name": fc_step.name,
        "call_id": fc_step.id,
        "result": [{"type": "text", "text": json.dumps(result)}],
    }],
    tools=TOOLS,
)
print("Final answer:", final_interaction.output_text)

Model chose tool: calculator {'expression': '(15 * 4) + 7'}
Tool result: 67
Final answer: (15 * 4) + 7 = 67


## Task 3 — Build a Minimal Agent Loop

`run_agent()` below is the real loop: send message → check for a
`function_call` step → execute the tool → append a `function_result` →
repeat until the model returns plain text, with a `max_iterations`
safeguard against infinite loops. Tested on a task that genuinely needs 2
tool calls: comparing weather across two cities.

In [6]:
def execute_tool(name, args, log=print):
    """TASK 5 guardrail: every failure becomes a string result the model can
    see and react to, instead of an uncaught exception that crashes the run."""
    if name not in TOOL_FUNCTIONS:
        msg = f"Unknown tool \'{name}\'. Available tools: {list(TOOL_FUNCTIONS)}"
        log(f"  [OBSERVE] ERROR: {msg}")
        return msg
    try:
        result = TOOL_FUNCTIONS[name](**args)
        log(f"  [OBSERVE] {name}({args}) -> {result}")
        return json.dumps(result) if not isinstance(result, str) else result
    except Exception as e:
        msg = f"Tool \'{name}\' failed: {e}"
        log(f"  [OBSERVE] ERROR: {msg}")
        return msg


SYSTEM_NOTE = (
    "You are a careful assistant with access to tools. Reason step by step. "
    "Only call a tool when it is actually needed to answer the user. "
    "When you have enough information, respond with a final plain-text answer "
    "and do not call any more tools.\n\n"
)


def run_agent(client, user_message, model="gemini-3.8-flash", max_iterations=6, verbose=True):
    """
    TASK 4 note on memory:
      - `history` IS the conversation memory — sent back in full on every
        call, since the API itself is stateless.
      - `working_memory` is a separate scratchpad WE keep, tracking every
        tool call and result, independent of what the model says in text —
        used for logging/debugging and for code that reasons about progress.
    """
    def log(msg):
        if verbose:
            print(msg)

    history = [{
        "type": "user_input",
        "content": [{"type": "text", "text": SYSTEM_NOTE + user_message}],
    }]
    working_memory = {"tool_calls": [], "observations": []}

    log(f"[USER] {user_message}\n")

    for i in range(1, max_iterations + 1):
        log(f"--- iteration {i} ---")

        interaction = client.interactions.create(
            model=model, store=False, input=history, tools=TOOLS,
        )

        for step in interaction.steps:
            history.append(step.model_dump())
            if getattr(step, "type", None) == "text" and getattr(step, "content", None):
                for part in step.content:
                    if getattr(part, "text", None):
                        log(f"  [REASON] {part.text.strip()}")

        fc_steps = [s for s in interaction.steps if s.type == "function_call"]

        if not fc_steps:
            final_text = interaction.output_text
            log(f"\n[FINAL ANSWER] {final_text}")
            return final_text, working_memory

        for fc in fc_steps:
            log(f"  [ACT] call {fc.name}({fc.arguments})")
            working_memory["tool_calls"].append({"name": fc.name, "input": fc.arguments})
            result_text = execute_tool(fc.name, fc.arguments, log=log)
            working_memory["observations"].append(result_text)
            history.append({
                "type": "function_result", "name": fc.name, "call_id": fc.id,
                "result": [{"type": "text", "text": result_text}],
            })

    log(f"\n[GUARDRAIL] max_iterations ({max_iterations}) reached without a final answer.")
    return None, working_memory


# --- Test: multi-step task requiring 2+ tool calls ---
final, mem = run_agent(client, "Look up the weather in Lahore and Karachi and tell me which is warmer.")
print("\nworking_memory:", mem)

[USER] Look up the weather in Lahore and Karachi and tell me which is warmer.

--- iteration 1 ---
  [ACT] call get_weather({'city': 'Lahore'})
  [OBSERVE] get_weather({'city': 'Lahore'}) -> {'city': 'Lahore', 'temp_c': 34, 'condition': 'Sunny'}
  [ACT] call get_weather({'city': 'Karachi'})
  [OBSERVE] get_weather({'city': 'Karachi'}) -> {'city': 'Karachi', 'temp_c': 29, 'condition': 'Humid'}
--- iteration 2 ---

[FINAL ANSWER] The current weather is:
- **Lahore**: 34°C, Sunny
- **Karachi**: 29°C, Humid

**Lahore** is warmer than Karachi (by 5°C).

working_memory: {'tool_calls': [{'name': 'get_weather', 'input': {'city': 'Lahore'}}, {'name': 'get_weather', 'input': {'city': 'Karachi'}}], 'observations': ['{"city": "Lahore", "temp_c": 34, "condition": "Sunny"}', '{"city": "Karachi", "temp_c": 29, "condition": "Humid"}']}


## Task 4 — Memory & State Handling

**Conversation memory** = the `history` list resent on every call (the API
is stateless — if it isn't in `history`, the model has no idea it
happened). **Working memory** = the separate `working_memory` scratchpad
the harness keeps — every tool call and result, tracked independently of
what the model chooses to say, so the surrounding code (logging, a UI,
loop detection) can reason about progress without re-parsing conversation
text.

Logging is already built in above — every `[REASON]` / `[ACT]` / `[OBSERVE]`
line printed during the run *is* that debugging habit. This is the most
useful piece of scaffolding to carry into every framework from tomorrow
onward — frameworks often hide this trace behind an abstraction, and
turning verbose logging back on is usually the first thing worth doing
when debugging one.

## Task 5 — Failure Modes & Guardrails

Deliberately breaking the agent, run for real:

In [7]:
# 1. Ambiguous request — no city given
print("=== Ambiguous request ===")
run_agent(client, "What's the weather like?")

=== Ambiguous request ===
[USER] What's the weather like?

--- iteration 1 ---

[FINAL ANSWER] Which city or location would you like to know the weather for?


('Which city or location would you like to know the weather for?',
 {'tool_calls': [], 'observations': []})

In [8]:
# 2. Tool error — city not in the stub dataset
print("=== Tool error ===")
run_agent(client, "What's the weather in Faisalabad?")

=== Tool error ===
[USER] What's the weather in Faisalabad?

--- iteration 1 ---
  [ACT] call get_weather({'city': 'Faisalabad'})
  [OBSERVE] ERROR: Tool 'get_weather' failed: "No weather data available for 'Faisalabad' (stub only covers a fixed city list)"
--- iteration 2 ---

[FINAL ANSWER] I cannot retrieve the current weather for Faisalabad because weather data for that city is not available in the demo system.


('I cannot retrieve the current weather for Faisalabad because weather data for that city is not available in the demo system.',
 {'tool_calls': [{'name': 'get_weather', 'input': {'city': 'Faisalabad'}}],
  'observations': ['Tool \'get_weather\' failed: "No weather data available for \'Faisalabad\' (stub only covers a fixed city list)"']})

In [9]:
# 3. Task needs a tool that was never defined
print("=== Undefined tool ===")
run_agent(client, "Send an email to my professor asking for an extension.")

=== Undefined tool ===
[USER] Send an email to my professor asking for an extension.

--- iteration 1 ---

[FINAL ANSWER] I don't have access to an email tool to send the message directly, but here is a professional draft you can copy, customize with your details, and send to your professor:

---

**Subject:** Extension Request: [Course Name / Number] – [Your Full Name]

Dear Professor [Professor's Last Name],

I hope this email finds you well. 

I am writing to respectfully request a short extension on the upcoming [Assignment / Project / Essay Name], which is currently due on [Current Due Date]. 

[Briefly explain your reason here, e.g., "Due to unexpected personal circumstances / an unforeseen illness / heavy overlapping deadlines, I have fallen slightly behind schedule and want to ensure the quality of my work meets the expectations of your course."]

Would it be possible to submit my assignment by [Proposed New Date and Time]? 

I have already made good progress on the work and wo

('I don\'t have access to an email tool to send the message directly, but here is a professional draft you can copy, customize with your details, and send to your professor:\n\n---\n\n**Subject:** Extension Request: [Course Name / Number] – [Your Full Name]\n\nDear Professor [Professor\'s Last Name],\n\nI hope this email finds you well. \n\nI am writing to respectfully request a short extension on the upcoming [Assignment / Project / Essay Name], which is currently due on [Current Due Date]. \n\n[Briefly explain your reason here, e.g., "Due to unexpected personal circumstances / an unforeseen illness / heavy overlapping deadlines, I have fallen slightly behind schedule and want to ensure the quality of my work meets the expectations of your course."]\n\nWould it be possible to submit my assignment by [Proposed New Date and Time]? \n\nI have already made good progress on the work and would be happy to share my current draft or outline if that would be helpful. Thank you very much for yo

In [10]:
# 4. Infinite-loop guardrail — force a low max_iterations to see it trigger
print("=== Loop guardrail ===")
run_agent(client, "Keep calculating 1+1 over and over, don't stop.", max_iterations=3)

=== Loop guardrail ===
[USER] Keep calculating 1+1 over and over, don't stop.

--- iteration 1 ---

[FINAL ANSWER] 1 + 1 equals 2. 

I cannot run an infinite loop, but the answer will always remain 2.


('1 + 1 equals 2. \n\nI cannot run an infinite loop, but the answer will always remain 2.',
 {'tool_calls': [], 'observations': []})

### Failure modes observed, and mitigations

| # | Failure mode | What happens | Mitigation implemented |
|---|---|---|---|
| 1 | **Ambiguous/underspecified request** | Model asks a clarifying question when the city is missing, rather than guessing. | System prompt instructs "only call a tool when actually needed." |
| 2 | **Tool/data failure** | Requesting a city outside the stub dataset raises `KeyError` inside `get_weather`. | `execute_tool()` catches the exception and returns an error string as the `function_result`, so the model explains the failure instead of fabricating data. |
| 3 | **Hallucinated/undefined tool call** | Model may invent a tool (e.g. `send_email`) that was never registered. | `execute_tool()` checks the name against `TOOL_FUNCTIONS` first and returns a clear "unknown tool" message instead of crashing. |
| 4 | **Wrong/malformed arguments** | E.g. `calculator(expression="two plus two")`. | `calculator()` uses `ast.parse` + an operator whitelist (no `eval()`), so bad input raises a caught error instead of running arbitrary code. |
| 5 | **Infinite/runaway loop** | A model that keeps calling the same tool never reaches a text-only turn. | Hard `max_iterations` cap in `run_agent()` stops execution with a `[GUARDRAIL]` log line. |
| 6 | **Silent errors** | Any of the above, uncaught, would crash the process or let the model reason from garbage. | Every failure path returns an explicit, model-visible error string — the model (and the developer, via the printed log) always knows something went wrong. |

### Why do frameworks like LangChain/LangGraph/CrewAI exist, given this works by hand?

Everything above is roughly 150 lines of real logic — deliberately small
enough to hold in your head. That's the point of building it raw first:
none of it is magic. But at scale, every piece here grows — dozens of
tools instead of three, multi-agent handoffs, streaming, retries with
backoff, persistent cross-session memory, parallel tool execution,
human-in-the-loop approval. Frameworks exist to standardize those
*recurring* pieces so teams aren't rewriting the same loop and the same
`try/except` around every tool call in every project. Having built it by
hand first means their abstractions now read as named versions of things
already written, not opaque magic — which is what makes debugging a
framework-based agent tractable when it misbehaves.